# Processamento de Sinais I — Aula Prática 2
## Questão 1 — Espectro de sinais cossenoidais

Sinal analisado:
$$x(t) = cos(2\pi f t)$$

Parâmetros:
- Frequência de amostragem: 44.1 kHz
- Frequências testadas: 500 Hz, 5000 Hz, 10000 Hz e 50000 Hz
- Objetivo: calcular o espectro e comentar os resultados.

## Importar bibliotecas e a função local de espectro

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from scipy.io import wavfile
from scipy import signal

def load_calculate_spectrum():
    notebook_path = Path('../tools/calculate_spectrum.ipynb')
    notebook = json.loads(notebook_path.read_text(encoding='utf-8'))

    namespace = {}
    for cell in notebook['cells']:
        if cell.get('cell_type') != 'code':
            continue

        source = ''.join(cell.get('source', []))
        exec(source, namespace)
        if 'calculate_spectrum' in namespace:
            return namespace['calculate_spectrum']

    raise RuntimeError('calculate_spectrum() nao encontrada em ../tools/calculate_spectrum.ipynb')

calculate_spectrum = load_calculate_spectrum()

plt.style.use('seaborn-v0_8-whitegrid')

def to_float_mono(data):
    data = np.asarray(data)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if np.issubdtype(data.dtype, np.integer):
        data = data.astype(np.float64) / np.iinfo(data.dtype).max
    else:
        data = data.astype(np.float64)
    peak = np.max(np.abs(data))
    if peak > 1:
        data = data / peak
    return data


def show_audio(audio, rate, label):
    print(label)
    display(Audio(audio, rate=rate))


def plot_spectrum(signal_in, sampling_frequency, title, max_frequency=None):
    freqs, amps = calculate_spectrum(signal_in, sampling_frequency, single_sided=True)
    plt.figure(figsize=(12, 4))
    plt.plot(freqs, amps)
    if max_frequency is not None:
        plt.xlim(0, max_frequency)
    plt.title(title)
    plt.xlabel('Frequencia (Hz)')
    plt.ylabel('Amplitude')
    plt.tight_layout()
    return freqs, amps


## Gerar os sinais e calcular os espectros

In [ ]:
fs = 44_100
frequencias = [500, 5_000, 10_000, 50_000]
duracao = 0.02

t = np.linspace(0, duracao, int(fs * duracao), endpoint=False)
sinais = {f: np.cos(2 * np.pi * f * t) for f in frequencias}

fig, axes = plt.subplots(len(frequencias), 2, figsize=(14, 12))

for i, f in enumerate(frequencias):
    axes[i, 0].plot(t[:400], sinais[f][:400])
    axes[i, 0].set_title(f'Sinal no tempo - f = {f} Hz')
    axes[i, 0].set_xlabel('Tempo (s)')
    axes[i, 0].set_ylabel('Amplitude')

    freqs, amps = calculate_spectrum(sinais[f], fs, single_sided=True)
    axes[i, 1].plot(freqs, amps)
    axes[i, 1].set_xlim(0, fs / 2)
    axes[i, 1].set_title(f'Espectro - f = {f} Hz')
    axes[i, 1].set_xlabel('Frequencia (Hz)')
    axes[i, 1].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()


## Comentários

Os sinais de 500 Hz, 5000 Hz e 10000 Hz apresentam um pico espectral nas próprias frequências, como esperado para uma cossenoide pura.

No caso de 50000 Hz, a frequência está acima da frequência de Nyquist ($f_s/2 = 22050$ Hz), então ocorre aliasing. O pico observado no espectro aparece na frequência refletida dentro da banda amostrada, mostrando que o sinal digital não representa corretamente a frequência analógica original.